# Brady Corporation Lumber Problem with AMPL in Python
## AMPL Modeling for Problem 6

## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

```python
%pip install amplpy
```

In [1]:
from __future__ import annotations

from functools import wraps
from time import perf_counter

In [2]:
%pip install amplpy
from amplpy import AMPL, modules

modules.install('highs')


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
def create_ampl_instance(solver: str = "highs"):
    ampl = AMPL()
    ampl.option["solver"] = solver
    ampl.option["solver_msg"] = 0
    return ampl

## Problem: Brady Corporation Weekly Lumber Cost Minimization

Brady needs 90,000 cubic feet of processed lumber per week.

**Sources:**
- **Grade 1 lumber:** $3/cuft purchase + $4/cuft drying = $7/cuft. Yields 0.7 cuft useful. Max 40,000 cuft/week.
- **Grade 2 lumber:** $7/cuft purchase + $4/cuft drying = $11/cuft. Yields 0.9 cuft useful. Max 60,000 cuft/week.
- **Own logs:** $3/cuft cutting + $2.50/cuft sawmill + $4/cuft drying = $9.50/cuft. Yields 0.8 cuft useful.

**Constraints:**
- Sawmill capacity: 35,000 cuft/week
- Drying time: 40 hours/week (144,000 seconds)
  - Grade 1: 1.2 sec/cuft, Grade 2: 0.8 sec/cuft, Logs: 1.3 sec/cuft

Minimize weekly cost.

In [4]:
@timed
def solve_brady(solver: str = "highs"):
    ampl = create_ampl_instance(solver)

    ampl.eval(
        r'''
        var g1 >= 0;
        var g2 >= 0;
        var logs >= 0;

        minimize TotalCost:
            7 * g1 + 11 * g2 + 9.50 * logs;

        subject to Demand:
            0.7 * g1 + 0.9 * g2 + 0.8 * logs >= 90000;

        subject to SawmillCapacity:
            logs <= 35000;

        subject to Grade1Supply:
            g1 <= 40000;

        subject to Grade2Supply:
            g2 <= 60000;

        subject to DryingTime:
            1.2 * g1 + 0.8 * g2 + 1.3 * logs <= 144000;
        '''
    )
    ampl.solve(solver=solver)

    solution = {
        "g1": round(ampl.get_value("g1"), 4),
        "g2": round(ampl.get_value("g2"), 4),
        "logs": round(ampl.get_value("logs"), 4),
        "cost": round(ampl.get_value("TotalCost"), 4),
    }
    return solution

In [5]:
result, elapsed = solve_brady()

print("=== BRADY CORPORATION RESULTS ===")
print(f"Grade 1 lumber purchased -> {result['g1']:.2f} cuft")
print(f"Grade 2 lumber purchased -> {result['g2']:.2f} cuft")
print(f"Own logs processed       -> {result['logs']:.2f} cuft")
print(f"Minimum weekly cost      -> ${result['cost']:,.2f}")
print(f"Time                     -> {elapsed:.8f} seconds")
print()

useful = 0.7*result['g1'] + 0.9*result['g2'] + 0.8*result['logs']
drying = 1.2*result['g1'] + 0.8*result['g2'] + 1.3*result['logs']
print("Verification:")
print(f"  Useful lumber:  {useful:,.1f} cuft (need >= 90,000)")
print(f"  Drying time:    {drying:,.1f} sec of {144000:,} sec ({drying/144000*100:.1f}%)")
print(f"  Sawmill usage:  {result['logs']:,.1f} of 35,000 ({result['logs']/35000*100:.1f}%)")
print()
print("Cost per useful cuft:")
if result['g1'] > 0:
    print(f"  Grade 1: ${7/0.7:.2f}/useful cuft")
if result['g2'] > 0:
    print(f"  Grade 2: ${11/0.9:.2f}/useful cuft")
if result['logs'] > 0:
    print(f"  Logs:    ${9.50/0.8:.2f}/useful cuft")

HiGHS 1.11.0=== BRADY CORPORATION RESULTS ===
Grade 1 lumber purchased -> 40000.00 cuft
Grade 2 lumber purchased -> 37777.78 cuft
Own logs processed       -> 35000.00 cuft
Minimum weekly cost      -> $1,028,055.56
Time                     -> 0.01191564 seconds

Verification:
  Useful lumber:  90,000.0 cuft (need >= 90,000)
  Drying time:    123,722.2 sec of 144,000 sec (85.9%)
  Sawmill usage:  35,000.0 of 35,000 (100.0%)

Cost per useful cuft:
  Grade 1: $10.00/useful cuft
  Grade 2: $12.22/useful cuft
  Logs:    $11.88/useful cuft


## Sensitivity Analysis

In [6]:
ampl = create_ampl_instance()

ampl.eval(
    r'''
    var g1 >= 0;
    var g2 >= 0;
    var logs >= 0;

    minimize TotalCost:
        7 * g1 + 11 * g2 + 9.50 * logs;

    subject to Demand:
        0.7 * g1 + 0.9 * g2 + 0.8 * logs >= 90000;

    subject to SawmillCapacity:
        logs <= 35000;

    subject to Grade1Supply:
        g1 <= 40000;

    subject to Grade2Supply:
        g2 <= 60000;

    subject to DryingTime:
        1.2 * g1 + 0.8 * g2 + 1.3 * logs <= 144000;
    '''
)
ampl.solve()

constraints = ["Demand", "SawmillCapacity", "Grade1Supply", "Grade2Supply", "DryingTime"]

print("=== SHADOW PRICES AND SLACK ===")
print(f"{'Constraint':<20} {'Shadow Price':>14} {'Slack':>12}")
print("-" * 48)
for c in constraints:
    con = ampl.get_constraint(c)
    print(f"{c:<20} {con.dual():>14.4f} {con.slack():>12.4f}")

print()
print("=== REDUCED COSTS ===")
for v in ["g1", "g2", "logs"]:
    var = ampl.get_variable(v)
    print(f"{v:<6} RC = {var.rc():.4f}   Value = {var.value():.2f}")

HiGHS 1.11.0=== SHADOW PRICES AND SLACK ===
Constraint             Shadow Price        Slack
------------------------------------------------
Demand                      12.2222       0.0000
SawmillCapacity             -0.2778       0.0000
Grade1Supply                -1.5556       0.0000
Grade2Supply                 0.0000   22222.2222
DryingTime                   0.0000   20277.7778

=== REDUCED COSTS ===
g1     RC = 0.0000   Value = 40000.00
g2     RC = 0.0000   Value = 37777.78
logs   RC = 0.0000   Value = 35000.00


## Interpretation

The Brady problem reveals the trade-off between different lumber sources:

- **Demand shadow price**: Shows the marginal cost of producing one additional useful cubic foot. This is the true cost at the margin.
- **Sawmill capacity**: If binding, expanding sawmill capacity would allow cheaper lumber (own logs at $11.875/useful cuft vs Grade 2 at $12.22/useful cuft).
- **Drying time**: If binding, this is the bottleneck. The shadow price shows the value of one additional second of drying capacity.
- **Supply limits**: Shadow prices indicate whether Brady would benefit from finding additional suppliers.